In [2]:
import pandas as pd
import numpy as np

match_summary = pd.read_csv(
    "../data/processed/match_summary.csv"
)

print(match_summary.shape)
match_summary.head()

(1193, 65)


,match_id,Unnamed: 0,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_balls,team_wicket,new_batter,power_surge_start,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out
0,335982,141607,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,0,RT Ponting,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",RT Ponting,False
1,335983,141832,2008-04-19,T20,Indian Premier League,1,Chennai Super Kings,Kings XI Punjab,0,1,...,1,0,MEK Hussey,NaN,0,1,0,"('ML Hayden', 'PA Patel')",MEK Hussey,False
2,335984,142080,2008-04-19,T20,Indian Premier League,1,Rajasthan Royals,Delhi Daredevils,0,1,...,1,0,SR Watson,NaN,0,1,0,"('T Kohli', 'YK Pathan')",SR Watson,False
3,335985,142299,2008-04-20,T20,Indian Premier League,1,Mumbai Indians,Royal Challengers Bangalore,0,1,...,1,0,DJ Thornely,NaN,0,1,0,"('L Ronchi', 'ST Jayasuriya')",DJ Thornely,False
4,335986,142545,2008-04-20,T20,Indian Premier League,1,Deccan Chargers,Kolkata Knight Riders,0,1,...,1,0,VVS Laxman,NaN,1,1,0,"('AC Gilchrist', 'Y Venugopal Rao')",VVS Laxman,False


In [3]:
team_mapping = {
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Rising Pune Supergiants": "Rising Pune Supergiants",
    "Rising Pune Supergiant": "Rising Pune Supergiants"
}

In [4]:
for col in ['toss_winner', 'match_won_by']:
    match_summary[col] = (
        match_summary[col]
        .replace(team_mapping)
    )

print(
    sorted(match_summary['match_won_by'].dropna().unique())
)

['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad', 'Unknown']


In [5]:
match_summary = match_summary[
    match_summary["match_won_by"] != "Unknown"
].copy()

print(match_summary.shape)

(1169, 65)


In [6]:
match_summary[
    ['match_won_by',
     'toss_winner',
     'season',
     'date']
].isnull().sum()

match_won_by    0
toss_winner     0
season          0
date            0
dtype: int64

In [7]:
match_summary['date'] = pd.to_datetime(
    match_summary['date']
)

match_summary = (
    match_summary
    .sort_values('date')
    .reset_index(drop=True)
)

match_summary[['date']].head()

,date
0,2008-04-18
1,2008-04-19
2,2008-04-19
3,2008-04-20
4,2008-04-20


In [8]:
match_summary[['date','match_won_by']].head(10)

,date,match_won_by
0,2008-04-18,Kolkata Knight Riders
1,2008-04-19,Chennai Super Kings
2,2008-04-19,Delhi Capitals
3,2008-04-20,Royal Challengers Bengaluru
4,2008-04-20,Kolkata Knight Riders
5,2008-04-21,Rajasthan Royals
6,2008-04-22,Delhi Capitals
7,2008-04-23,Chennai Super Kings
8,2008-04-24,Rajasthan Royals
9,2008-04-25,Punjab Kings


In [9]:
teams = sorted(match_summary['match_won_by'].unique())

print(f"Number of teams: {len(teams)}")
teams

Number of teams: 15


['Chennai Super Kings',
 'Deccan Chargers',
 'Delhi Capitals',
 'Gujarat Lions',
 'Gujarat Titans',
 'Kochi Tuskers Kerala',
 'Kolkata Knight Riders',
 'Lucknow Super Giants',
 'Mumbai Indians',
 'Pune Warriors',
 'Punjab Kings',
 'Rajasthan Royals',
 'Rising Pune Supergiants',
 'Royal Challengers Bengaluru',
 'Sunrisers Hyderabad']

In [10]:
team_wins = {team: 0 for team in teams}
team_matches = {team: 0 for team in teams}

In [11]:
historical_win_pct = []

for _, row in match_summary.iterrows():

    winner = row['match_won_by']

    if team_matches[winner] == 0:
        historical_win_pct.append(0)
    else:
        historical_win_pct.append(
            team_wins[winner] / team_matches[winner]
        )

    team_matches[winner] += 1
    team_wins[winner] += 1

match_summary['winner_historical_win_pct'] = historical_win_pct

match_summary[
    ['date',
     'match_won_by',
     'winner_historical_win_pct']
].head(20)

,date,match_won_by,winner_historical_win_pct
0,2008-04-18,Kolkata Knight Riders,0.0
1,2008-04-19,Chennai Super Kings,0.0
2,2008-04-19,Delhi Capitals,0.0
3,2008-04-20,Royal Challengers Bengaluru,0.0
4,2008-04-20,Kolkata Knight Riders,1.0
5,2008-04-21,Rajasthan Royals,0.0
6,2008-04-22,Delhi Capitals,1.0
7,2008-04-23,Chennai Super Kings,1.0
8,2008-04-24,Rajasthan Royals,1.0
9,2008-04-25,Punjab Kings,0.0


In [13]:
ipl = pd.read_csv("../data/raw/IPL.csv")

print(ipl.shape)

C:\Users\Om Mishra\AppData\Local\Temp\ipykernel_22256\2455846577.py:1: DtypeWarning: Columns (28,29,30,31,43,46,47,48,51) have mixed types. Specify dtype option on import or set low_memory=False.
  ipl = pd.read_csv("../data/raw/IPL.csv")


(283678, 65)


In [14]:
ipl.groupby('match_id')['batting_team'].nunique().value_counts()

batting_team
2    1187
1       6
Name: count, dtype: int64

In [15]:
ipl.groupby('match_id')['bowling_team'].nunique().value_counts()

bowling_team
2    1187
1       6
Name: count, dtype: int64

In [16]:
sample_match = 335982

ipl.loc[
    ipl['match_id'] == sample_match,
    ['batting_team', 'bowling_team']
].drop_duplicates()

,batting_team,bowling_team
0,Kolkata Knight Riders,Royal Challengers Bangalore
124,Royal Challengers Bangalore,Kolkata Knight Riders


In [17]:
match_teams = (
    ipl.groupby('match_id')['batting_team']
       .unique()
       .reset_index()
)

match_teams.head()

,match_id,batting_team
0,335982,"[Kolkata Knight Riders, Royal Challengers Bang..."
1,335983,"[Chennai Super Kings, Kings XI Punjab]"
2,335984,"[Rajasthan Royals, Delhi Daredevils]"
3,335985,"[Mumbai Indians, Royal Challengers Bangalore]"
4,335986,"[Deccan Chargers, Kolkata Knight Riders]"


In [18]:
match_teams['team1'] = match_teams['batting_team'].apply(
    lambda x: x[0] if len(x) >= 1 else None
)

match_teams['team2'] = match_teams['batting_team'].apply(
    lambda x: x[1] if len(x) >= 2 else None
)

match_teams.head()

,match_id,batting_team,team1,team2
0,335982,"[Kolkata Knight Riders, Royal Challengers Bang...",Kolkata Knight Riders,Royal Challengers Bangalore
1,335983,"[Chennai Super Kings, Kings XI Punjab]",Chennai Super Kings,Kings XI Punjab
2,335984,"[Rajasthan Royals, Delhi Daredevils]",Rajasthan Royals,Delhi Daredevils
3,335985,"[Mumbai Indians, Royal Challengers Bangalore]",Mumbai Indians,Royal Challengers Bangalore
4,335986,"[Deccan Chargers, Kolkata Knight Riders]",Deccan Chargers,Kolkata Knight Riders


In [19]:
model_df = match_summary.merge(
    match_teams[['match_id', 'team1', 'team2']],
    on='match_id',
    how='left'
)

model_df[
    [
        'match_id',
        'team1',
        'team2',
        'match_won_by'
    ]
].head()

,match_id,team1,team2,match_won_by
0,335982,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders
1,335983,Chennai Super Kings,Kings XI Punjab,Chennai Super Kings
2,335984,Rajasthan Royals,Delhi Daredevils,Delhi Capitals
3,335985,Mumbai Indians,Royal Challengers Bangalore,Royal Challengers Bengaluru
4,335986,Deccan Chargers,Kolkata Knight Riders,Kolkata Knight Riders


In [20]:
print(model_df.shape)

model_df[
    [
        'match_id',
        'team1',
        'team2',
        'match_won_by'
    ]
].head(10)

(1169, 68)


,match_id,team1,team2,match_won_by
0,335982,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders
1,335983,Chennai Super Kings,Kings XI Punjab,Chennai Super Kings
2,335984,Rajasthan Royals,Delhi Daredevils,Delhi Capitals
3,335985,Mumbai Indians,Royal Challengers Bangalore,Royal Challengers Bengaluru
4,335986,Deccan Chargers,Kolkata Knight Riders,Kolkata Knight Riders
5,335987,Kings XI Punjab,Rajasthan Royals,Rajasthan Royals
6,335988,Deccan Chargers,Delhi Daredevils,Delhi Capitals
7,335989,Chennai Super Kings,Mumbai Indians,Chennai Super Kings
8,335990,Deccan Chargers,Rajasthan Royals,Rajasthan Royals
9,335991,Kings XI Punjab,Mumbai Indians,Punjab Kings


In [21]:
for col in ['team1', 'team2', 'match_won_by', 'toss_winner']:
    model_df[col] = model_df[col].replace(team_mapping)

In [22]:
model_df['team1_win'] = (
    model_df['team1'] == model_df['match_won_by']
).astype(int)

model_df[
    [
        'team1',
        'team2',
        'match_won_by',
        'team1_win'
    ]
].head(10)

,team1,team2,match_won_by,team1_win
0,Kolkata Knight Riders,Royal Challengers Bengaluru,Kolkata Knight Riders,1
1,Chennai Super Kings,Punjab Kings,Chennai Super Kings,1
2,Rajasthan Royals,Delhi Capitals,Delhi Capitals,0
3,Mumbai Indians,Royal Challengers Bengaluru,Royal Challengers Bengaluru,0
4,Deccan Chargers,Kolkata Knight Riders,Kolkata Knight Riders,0
5,Punjab Kings,Rajasthan Royals,Rajasthan Royals,0
6,Deccan Chargers,Delhi Capitals,Delhi Capitals,0
7,Chennai Super Kings,Mumbai Indians,Chennai Super Kings,1
8,Deccan Chargers,Rajasthan Royals,Rajasthan Royals,0
9,Punjab Kings,Mumbai Indians,Punjab Kings,1


In [23]:
model_df['team1_win'].value_counts()

team1_win
0    635
1    534
Name: count, dtype: int64

In [24]:
teams = sorted(
    set(model_df['team1'])
    .union(set(model_df['team2']))
)

elo_ratings = {
    team: 1500 for team in teams
}

K = 32

In [25]:
team1_elo = []
team2_elo = []

In [26]:
for _, row in model_df.iterrows():

    t1 = row['team1']
    t2 = row['team2']

    # Store PRE-MATCH ratings
    team1_elo.append(elo_ratings[t1])
    team2_elo.append(elo_ratings[t2])

    r1 = elo_ratings[t1]
    r2 = elo_ratings[t2]

    expected1 = 1 / (1 + 10 ** ((r2 - r1) / 400))
    expected2 = 1 - expected1

    actual1 = row['team1_win']
    actual2 = 1 - actual1

    elo_ratings[t1] += K * (actual1 - expected1)
    elo_ratings[t2] += K * (actual2 - expected2)

In [27]:
len(team1_elo), len(model_df)

(1169, 1169)

In [28]:
model_df['team1_elo'] = team1_elo
model_df['team2_elo'] = team2_elo

model_df['elo_diff'] = (
    model_df['team1_elo']
    - model_df['team2_elo']
)

In [29]:
model_df[
    [
        'team1',
        'team2',
        'team1_elo',
        'team2_elo',
        'elo_diff',
        'team1_win'
    ]
].head(15)

,team1,team2,team1_elo,team2_elo,elo_diff,team1_win
0,Kolkata Knight Riders,Royal Challengers Bengaluru,1500.000000,1500.000000,0.000000,1
1,Chennai Super Kings,Punjab Kings,1500.000000,1500.000000,0.000000,1
2,Rajasthan Royals,Delhi Capitals,1500.000000,1500.000000,0.000000,0
3,Mumbai Indians,Royal Challengers Bengaluru,1500.000000,1484.000000,16.000000,0
4,Deccan Chargers,Kolkata Knight Riders,1500.000000,1516.000000,-16.000000,0
5,Punjab Kings,Rajasthan Royals,1484.000000,1484.000000,0.000000,0
6,Deccan Chargers,Delhi Capitals,1484.736307,1516.000000,-31.263693,0
7,Chennai Super Kings,Mumbai Indians,1516.000000,1483.263693,32.736307,1
8,Deccan Chargers,Rajasthan Royals,1470.172180,1500.000000,-29.827820,0
9,Punjab Kings,Mumbai Indians,1468.000000,1468.766810,-0.766810,1


In [30]:
elo_table = pd.DataFrame(
    elo_ratings.items(),
    columns=['team', 'elo']
)

elo_table = elo_table.sort_values(
    'elo',
    ascending=False
)

elo_table.head(20)

,team,elo
13,Royal Challengers Bengaluru,1636.660422
10,Punjab Kings,1588.921060
12,Rising Pune Supergiants,1556.491328
4,Gujarat Titans,1546.489316
11,Rajasthan Royals,1528.311934
2,Delhi Capitals,1513.796955
14,Sunrisers Hyderabad,1505.372596
7,Lucknow Super Giants,1492.513853
8,Mumbai Indians,1486.197482
5,Kochi Tuskers Kerala,1476.989270


In [31]:
elo_table.tail(10)

,team,elo
2,Delhi Capitals,1513.796955
14,Sunrisers Hyderabad,1505.372596
7,Lucknow Super Giants,1492.513853
8,Mumbai Indians,1486.197482
5,Kochi Tuskers Kerala,1476.989270
0,Chennai Super Kings,1471.067596
6,Kolkata Knight Riders,1467.676785
3,Gujarat Lions,1453.735466
1,Deccan Chargers,1424.687605
9,Pune Warriors,1351.088332


In [32]:
model_df['match_won_by'].value_counts()

match_won_by
Mumbai Indians                 152
Chennai Super Kings            144
Royal Challengers Bengaluru    136
Kolkata Knight Riders          135
Punjab Kings                   123
Delhi Capitals                 120
Rajasthan Royals               118
Sunrisers Hyderabad             95
Gujarat Titans                  39
Lucknow Super Giants            32
Deccan Chargers                 29
Rising Pune Supergiants         15
Gujarat Lions                   13
Pune Warriors                   12
Kochi Tuskers Kerala             6
Name: count, dtype: int64

In [33]:
team_matches = pd.concat(
    [
        model_df['team1'],
        model_df['team2']
    ]
).value_counts()

team_matches

Mumbai Indians                 278
Royal Challengers Bengaluru    269
Kolkata Knight Riders          263
Punjab Kings                   262
Delhi Capitals                 262
Chennai Super Kings            255
Rajasthan Royals               234
Sunrisers Hyderabad            196
Deccan Chargers                 75
Gujarat Titans                  64
Lucknow Super Giants            62
Pune Warriors                   45
Rising Pune Supergiants         30
Gujarat Lions                   29
Kochi Tuskers Kerala            14
Name: count, dtype: int64

In [34]:
elo_table = elo_table.merge(
    team_matches.rename('matches'),
    left_on='team',
    right_index=True
)

elo_table.sort_values(
    'elo',
    ascending=False
)

,team,elo,matches
13,Royal Challengers Bengaluru,1636.660422,269
10,Punjab Kings,1588.921060,262
12,Rising Pune Supergiants,1556.491328,30
4,Gujarat Titans,1546.489316,64
11,Rajasthan Royals,1528.311934,234
2,Delhi Capitals,1513.796955,262
14,Sunrisers Hyderabad,1505.372596,196
7,Lucknow Super Giants,1492.513853,62
8,Mumbai Indians,1486.197482,278
5,Kochi Tuskers Kerala,1476.989270,14


In [35]:
model_df.to_csv(
    "../data/processed/model_df_with_elo.csv",
    index=False
)

print("Saved")

Saved


In [36]:
team_history = {
    team: []
    for team in teams
}

In [37]:
team1_form = []
team2_form = []

In [38]:
for _, row in model_df.iterrows():

    t1 = row['team1']
    t2 = row['team2']

    if len(team_history[t1]) == 0:
        team1_form.append(0.5)
    else:
        team1_form.append(
            np.mean(team_history[t1][-5:])
        )

    if len(team_history[t2]) == 0:
        team2_form.append(0.5)
    else:
        team2_form.append(
            np.mean(team_history[t2][-5:])
        )

    winner = row['match_won_by']

    if winner == t1:
        team_history[t1].append(1)
        team_history[t2].append(0)
    else:
        team_history[t1].append(0)
        team_history[t2].append(1)

In [39]:
len(team1_form), len(model_df)

(1169, 1169)

In [40]:
model_df['team1_form'] = team1_form
model_df['team2_form'] = team2_form

model_df['form_diff'] = (
    model_df['team1_form']
    - model_df['team2_form']
)

In [41]:
model_df[
    [
        'team1',
        'team2',
        'team1_elo',
        'team2_elo',
        'elo_diff',
        'team1_form',
        'team2_form',
        'form_diff',
        'team1_win'
    ]
].tail(15)

,team1,team2,team1_elo,team2_elo,elo_diff,team1_form,team2_form,form_diff,team1_win
1154,Rajasthan Royals,Gujarat Titans,1491.573552,1531.099976,-39.526424,0.4,0.2,0.2,1
1155,Sunrisers Hyderabad,Lucknow Super Giants,1517.193030,1485.090368,32.102662,0.8,0.2,0.6,0
1156,Royal Challengers Bengaluru,Chennai Super Kings,1627.507073,1441.245366,186.261707,0.8,0.4,0.4,1
1157,Rajasthan Royals,Mumbai Indians,1509.385998,1526.790782,-17.404784,0.6,0.4,0.2,1
1158,Gujarat Titans,Delhi Capitals,1513.287529,1552.112404,-38.824875,0.0,0.6,-0.6,1
1159,Kolkata Knight Riders,Lucknow Super Giants,1501.054016,1502.564557,-1.510542,0.2,0.4,-0.2,0
1160,Royal Challengers Bengaluru,Rajasthan Royals,1635.666538,1526.186848,109.479690,1.0,0.8,0.2,0
1161,Sunrisers Hyderabad,Punjab Kings,1499.718841,1563.341433,-63.622592,0.6,0.6,0.0,0
1162,Chennai Super Kings,Delhi Capitals,1433.085900,1534.331858,-101.245958,0.2,0.6,-0.4,1
1163,Lucknow Super Giants,Gujarat Titans,1518.494995,1531.068076,-12.573081,0.6,0.2,0.4,0


In [42]:
model_df[
    [
        'team1_win',
        'elo_diff',
        'form_diff'
    ]
].corr()

,team1_win,elo_diff,form_diff
team1_win,1.000000,0.050354,0.022572
elo_diff,0.050354,1.000000,0.615696
form_diff,0.022572,0.615696,1.000000


In [43]:
model_df['team1_won_toss'] = (
    model_df['team1'] == model_df['toss_winner']
).astype(int)

In [44]:
model_df['team2_won_toss'] = (
    model_df['team2'] == model_df['toss_winner']
).astype(int)

In [45]:
model_df[
    [
        'team1',
        'team2',
        'toss_winner',
        'team1_won_toss',
        'team2_won_toss'
    ]
].head(10)

,team1,team2,toss_winner,team1_won_toss,team2_won_toss
0,Kolkata Knight Riders,Royal Challengers Bengaluru,Royal Challengers Bengaluru,0,1
1,Chennai Super Kings,Punjab Kings,Chennai Super Kings,1,0
2,Rajasthan Royals,Delhi Capitals,Rajasthan Royals,1,0
3,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,1,0
4,Deccan Chargers,Kolkata Knight Riders,Deccan Chargers,1,0
5,Punjab Kings,Rajasthan Royals,Punjab Kings,1,0
6,Deccan Chargers,Delhi Capitals,Deccan Chargers,1,0
7,Chennai Super Kings,Mumbai Indians,Mumbai Indians,0,1
8,Deccan Chargers,Rajasthan Royals,Rajasthan Royals,0,1
9,Punjab Kings,Mumbai Indians,Mumbai Indians,0,1


In [46]:
model_df[
    [
        'team1_win',
        'team1_won_toss'
    ]
].corr()

,team1_win,team1_won_toss
team1_win,1.00000,0.00961
team1_won_toss,0.00961,1.00000


In [47]:
[
    col for col in model_df.columns
    if col in [
        'venue',
        'city',
        'season',
        'match_type',
        'toss_decision',
        'event_name'
    ]
]

['match_type', 'event_name', 'toss_decision', 'venue', 'city', 'season']

In [48]:
model_df[
    [
        'venue',
        'city',
        'season',
        'toss_decision'
    ]
].head()

,venue,city,season,toss_decision
0,M Chinnaswamy Stadium,Bangalore,2007/08,field
1,"Punjab Cricket Association Stadium, Mohali",Chandigarh,2007/08,bat
2,Feroz Shah Kotla,Delhi,2007/08,bat
3,Wankhede Stadium,Mumbai,2007/08,bat
4,Eden Gardens,Kolkata,2007/08,bat


In [49]:
model_df['venue'].nunique()

59

In [50]:
innings_summary = (
    ipl.groupby(
        [
            'match_id',
            'innings',
            'batting_team'
        ]
    )
    .agg(
        total_runs=('runs_total', 'sum'),
        wickets=('wicket_kind', lambda x: x.notna().sum())
    )
    .reset_index()
)

innings_summary.head()

,match_id,innings,batting_team,total_runs,wickets
0,335982,1,Kolkata Knight Riders,222,3
1,335982,2,Royal Challengers Bangalore,82,10
2,335983,1,Chennai Super Kings,240,5
3,335983,2,Kings XI Punjab,207,4
4,335984,1,Rajasthan Royals,129,8


In [51]:
innings_summary.shape

(2412, 5)

In [52]:
innings_summary.head(20)

,match_id,innings,batting_team,total_runs,wickets
0,335982,1,Kolkata Knight Riders,222,3
1,335982,2,Royal Challengers Bangalore,82,10
2,335983,1,Chennai Super Kings,240,5
3,335983,2,Kings XI Punjab,207,4
4,335984,1,Rajasthan Royals,129,8
5,335984,2,Delhi Daredevils,132,1
6,335985,1,Mumbai Indians,165,7
7,335985,2,Royal Challengers Bangalore,166,5
8,335986,1,Deccan Chargers,110,10
9,335986,2,Kolkata Knight Riders,112,5


In [53]:
innings_summary.to_csv(
    "../data/processed/innings_summary.csv",
    index=False
)

In [54]:
team_match_stats = (
    innings_summary[
        [
            'match_id',
            'batting_team',
            'total_runs',
            'wickets'
        ]
    ]
    .copy()
)

team_match_stats.head()

,match_id,batting_team,total_runs,wickets
0,335982,Kolkata Knight Riders,222,3
1,335982,Royal Challengers Bangalore,82,10
2,335983,Chennai Super Kings,240,5
3,335983,Kings XI Punjab,207,4
4,335984,Rajasthan Royals,129,8


In [55]:
team_match_stats = team_match_stats.merge(
    model_df[
        [
            'match_id',
            'date'
        ]
    ],
    on='match_id',
    how='left'
)

team_match_stats.head()

,match_id,batting_team,total_runs,wickets,date
0,335982,Kolkata Knight Riders,222,3,2008-04-18
1,335982,Royal Challengers Bangalore,82,10,2008-04-18
2,335983,Chennai Super Kings,240,5,2008-04-19
3,335983,Kings XI Punjab,207,4,2008-04-19
4,335984,Rajasthan Royals,129,8,2008-04-19


In [56]:
team_match_stats = (
    team_match_stats
    .sort_values('date')
    .reset_index(drop=True)
)

team_match_stats.head()

,match_id,batting_team,total_runs,wickets,date
0,335982,Kolkata Knight Riders,222,3,2008-04-18
1,335982,Royal Challengers Bangalore,82,10,2008-04-18
2,335983,Chennai Super Kings,240,5,2008-04-19
3,335983,Kings XI Punjab,207,4,2008-04-19
4,335984,Rajasthan Royals,129,8,2008-04-19


In [57]:
team_match_stats.shape

(2412, 5)

In [58]:
team_match_stats.head(10)

,match_id,batting_team,total_runs,wickets,date
0,335982,Kolkata Knight Riders,222,3,2008-04-18
1,335982,Royal Challengers Bangalore,82,10,2008-04-18
2,335983,Chennai Super Kings,240,5,2008-04-19
3,335983,Kings XI Punjab,207,4,2008-04-19
4,335984,Rajasthan Royals,129,8,2008-04-19
5,335984,Delhi Daredevils,132,1,2008-04-19
6,335985,Mumbai Indians,165,7,2008-04-20
7,335985,Royal Challengers Bangalore,166,5,2008-04-20
8,335986,Deccan Chargers,110,10,2008-04-20
9,335986,Kolkata Knight Riders,112,5,2008-04-20


In [59]:
team_match_stats['batting_team'] = (
    team_match_stats['batting_team']
    .replace(team_mapping)
)

In [60]:
sorted(
    team_match_stats['batting_team']
    .unique()
)

['Chennai Super Kings',
 'Deccan Chargers',
 'Delhi Capitals',
 'Gujarat Lions',
 'Gujarat Titans',
 'Kochi Tuskers Kerala',
 'Kolkata Knight Riders',
 'Lucknow Super Giants',
 'Mumbai Indians',
 'Pune Warriors',
 'Punjab Kings',
 'Rajasthan Royals',
 'Rising Pune Supergiants',
 'Royal Challengers Bengaluru',
 'Sunrisers Hyderabad']

In [61]:
sorted(
    team_match_stats['batting_team']
    .unique()
)

['Chennai Super Kings',
 'Deccan Chargers',
 'Delhi Capitals',
 'Gujarat Lions',
 'Gujarat Titans',
 'Kochi Tuskers Kerala',
 'Kolkata Knight Riders',
 'Lucknow Super Giants',
 'Mumbai Indians',
 'Pune Warriors',
 'Punjab Kings',
 'Rajasthan Royals',
 'Rising Pune Supergiants',
 'Royal Challengers Bengaluru',
 'Sunrisers Hyderabad']

In [62]:
sorted(
    team_match_stats['batting_team']
    .unique()
)

['Chennai Super Kings',
 'Deccan Chargers',
 'Delhi Capitals',
 'Gujarat Lions',
 'Gujarat Titans',
 'Kochi Tuskers Kerala',
 'Kolkata Knight Riders',
 'Lucknow Super Giants',
 'Mumbai Indians',
 'Pune Warriors',
 'Punjab Kings',
 'Rajasthan Royals',
 'Rising Pune Supergiants',
 'Royal Challengers Bengaluru',
 'Sunrisers Hyderabad']

In [64]:
team_runs_history = {
    team: []
    for team in team_match_stats['batting_team'].unique()
}

avg_runs_last5 = []

In [65]:
for _, row in team_match_stats.iterrows():

    team = row['batting_team']

    history = team_runs_history[team]

    if len(history) == 0:
        avg_runs_last5.append(
            team_match_stats['total_runs'].mean()
        )
    else:
        avg_runs_last5.append(
            np.mean(history[-5:])
        )

    history.append(row['total_runs'])

In [66]:
team_match_stats['avg_runs_last5'] = avg_runs_last5

In [67]:
team_match_stats[
    [
        'date',
        'batting_team',
        'total_runs',
        'avg_runs_last5'
    ]
].head(20)

,date,batting_team,total_runs,avg_runs_last5
0,2008-04-18,Kolkata Knight Riders,222,158.727197
1,2008-04-18,Royal Challengers Bengaluru,82,158.727197
2,2008-04-19,Chennai Super Kings,240,158.727197
3,2008-04-19,Punjab Kings,207,158.727197
4,2008-04-19,Rajasthan Royals,129,158.727197
5,2008-04-19,Delhi Capitals,132,158.727197
6,2008-04-20,Mumbai Indians,165,158.727197
7,2008-04-20,Royal Challengers Bengaluru,166,82.000000
8,2008-04-20,Deccan Chargers,110,158.727197
9,2008-04-20,Kolkata Knight Riders,112,222.000000


In [68]:
team_wickets_history = {
    team: []
    for team in team_match_stats['batting_team'].unique()
}

avg_wickets_last5 = []

In [69]:
for _, row in team_match_stats.iterrows():

    team = row['batting_team']

    history = team_wickets_history[team]

    if len(history) == 0:
        avg_wickets_last5.append(
            team_match_stats['wickets'].mean()
        )
    else:
        avg_wickets_last5.append(
            np.mean(history[-5:])
        )

    history.append(row['wickets'])

In [70]:
team_match_stats['avg_wickets_last5'] = avg_wickets_last5

team_match_stats[
    [
        'batting_team',
        'wickets',
        'avg_wickets_last5'
    ]
].head(20)

,batting_team,wickets,avg_wickets_last5
0,Kolkata Knight Riders,3,5.848259
1,Royal Challengers Bengaluru,10,5.848259
2,Chennai Super Kings,5,5.848259
3,Punjab Kings,4,5.848259
4,Rajasthan Royals,8,5.848259
5,Delhi Capitals,1,5.848259
6,Mumbai Indians,7,5.848259
7,Royal Challengers Bengaluru,5,10.000000
8,Deccan Chargers,10,5.848259
9,Kolkata Knight Riders,5,3.000000


In [71]:
team_match_stats['avg_wickets_last5'] = avg_wickets_last5

team_match_stats[
    [
        'batting_team',
        'wickets',
        'avg_wickets_last5'
    ]
].head(20)

,batting_team,wickets,avg_wickets_last5
0,Kolkata Knight Riders,3,5.848259
1,Royal Challengers Bengaluru,10,5.848259
2,Chennai Super Kings,5,5.848259
3,Punjab Kings,4,5.848259
4,Rajasthan Royals,8,5.848259
5,Delhi Capitals,1,5.848259
6,Mumbai Indians,7,5.848259
7,Royal Challengers Bengaluru,5,10.000000
8,Deccan Chargers,10,5.848259
9,Kolkata Knight Riders,5,3.000000


In [73]:
team_features = team_match_stats[
    [
        'match_id',
        'batting_team',
        'avg_runs_last5',
        'avg_wickets_last5'
    ]
].copy()

team_features.head()

,match_id,batting_team,avg_runs_last5,avg_wickets_last5
0,335982,Kolkata Knight Riders,158.727197,5.848259
1,335982,Royal Challengers Bengaluru,158.727197,5.848259
2,335983,Chennai Super Kings,158.727197,5.848259
3,335983,Punjab Kings,158.727197,5.848259
4,335984,Rajasthan Royals,158.727197,5.848259


In [74]:
team_features[
    team_features['match_id'] == 335982
]

,match_id,batting_team,avg_runs_last5,avg_wickets_last5
0,335982,Kolkata Knight Riders,158.727197,5.848259
1,335982,Royal Challengers Bengaluru,158.727197,5.848259


In [75]:
team_features.groupby(
    'match_id'
).size().value_counts()

2    1172
4      14
1       6
6       1
Name: count, dtype: int64

In [78]:
team_features.groupby(
    ['match_id', 'batting_team']
).agg(
    avg_runs_last5=('avg_runs_last5', 'last'),
    avg_wickets_last5=('avg_wickets_last5', 'last')
).reset_index()

,match_id,batting_team,avg_runs_last5,avg_wickets_last5
0,335982,Kolkata Knight Riders,158.727197,5.848259
1,335982,Royal Challengers Bengaluru,158.727197,5.848259
2,335983,Chennai Super Kings,158.727197,5.848259
3,335983,Punjab Kings,158.727197,5.848259
4,335984,Delhi Capitals,158.727197,5.848259
...,...,...,...,...
2375,1529265,Kolkata Knight Riders,181.800000,6.800000
2376,1529266,Lucknow Super Giants,174.800000,6.600000
2377,1529266,Royal Challengers Bengaluru,216.800000,5.600000
2378,1529267,Mumbai Indians,186.800000,6.200000


In [79]:
team_features_clean = (
    team_features
    .groupby(
        ['match_id', 'batting_team']
    )
    .agg(
        avg_runs_last5=('avg_runs_last5', 'last'),
        avg_wickets_last5=('avg_wickets_last5', 'last')
    )
    .reset_index()
)

In [80]:
team_features_clean.groupby(
    'match_id'
).size().value_counts()

2    1187
1       6
Name: count, dtype: int64

In [81]:
model_df = model_df.copy()

In [82]:
team_features_clean

,match_id,batting_team,avg_runs_last5,avg_wickets_last5
0,335982,Kolkata Knight Riders,158.727197,5.848259
1,335982,Royal Challengers Bengaluru,158.727197,5.848259
2,335983,Chennai Super Kings,158.727197,5.848259
3,335983,Punjab Kings,158.727197,5.848259
4,335984,Delhi Capitals,158.727197,5.848259
...,...,...,...,...
2375,1529265,Kolkata Knight Riders,181.800000,6.800000
2376,1529266,Lucknow Super Giants,174.800000,6.600000
2377,1529266,Royal Challengers Bengaluru,216.800000,5.600000
2378,1529267,Mumbai Indians,186.800000,6.200000


In [83]:
team_features_clean.groupby(
    'match_id'
).size().value_counts()

2    1187
1       6
Name: count, dtype: int64

In [84]:
model_df = model_df.merge(
    team_features_clean,
    left_on=['match_id', 'team1'],
    right_on=['match_id', 'batting_team'],
    how='left'
)

model_df = model_df.rename(
    columns={
        'avg_runs_last5': 'team1_avg_runs_last5',
        'avg_wickets_last5': 'team1_avg_wickets_last5'
    }
)

model_df = model_df.drop(
    columns=['batting_team']
)

In [85]:
model_df = model_df.merge(
    team_features_clean,
    left_on=['match_id', 'team2'],
    right_on=['match_id', 'batting_team'],
    how='left'
)

model_df = model_df.rename(
    columns={
        'avg_runs_last5': 'team2_avg_runs_last5',
        'avg_wickets_last5': 'team2_avg_wickets_last5'
    }
)

model_df = model_df.drop(
    columns=['batting_team']
)

In [87]:
model_df.columns[model_df.columns.duplicated()]

Index(['team1_avg_runs_last5', 'team1_avg_wickets_last5'], dtype='object')

In [88]:
model_df.shape

(1169, 84)

In [89]:
for col in model_df.columns:
    print(col)

match_id
Unnamed: 0
date
match_type
event_name
innings
batting_team_x
bowling_team
over
ball
ball_no
batter
bat_pos
runs_batter
balls_faced
bowler
valid_ball
runs_extras
runs_total
runs_bowler
runs_not_boundary
extra_type
non_striker
non_striker_pos
wicket_kind
player_out
fielders
runs_target
review_batter
team_reviewed
review_decision
umpire
umpires_call
player_of_match
match_won_by
win_outcome
toss_winner
toss_decision
venue
city
day
month
year
season
gender
team_type
superover_winner
result_type
method
balls_per_over
overs
event_match_no
stage
match_number
team_runs
team_balls
team_wicket
new_batter
power_surge_start
batter_runs
batter_balls
bowler_wicket
batting_partners
next_batter
striker_out
winner_historical_win_pct
team1
team2
team1_win
team1_elo
team2_elo
elo_diff
team1_form
team2_form
form_diff
team1_won_toss
team2_won_toss
batting_team_y
team1_avg_runs_last5
team1_avg_wickets_last5
team1_avg_runs_last5
team1_avg_wickets_last5
team2_avg_runs_last5
team2_avg_wickets_last5


In [90]:
model_df = model_df.loc[:, ~model_df.columns.duplicated()]

In [91]:
model_df.columns[model_df.columns.duplicated()]

Index([], dtype='object')

In [92]:
model_df.shape

(1169, 82)

In [93]:
[c for c in model_df.columns if 'runs' in c.lower()]

['runs_batter',
 'runs_extras',
 'runs_total',
 'runs_bowler',
 'runs_not_boundary',
 'runs_target',
 'team_runs',
 'batter_runs',
 'team1_avg_runs_last5',
 'team2_avg_runs_last5']

In [94]:
[c for c in model_df.columns if 'wicket' in c.lower()]

['wicket_kind',
 'team_wicket',
 'bowler_wicket',
 'team1_avg_wickets_last5',
 'team2_avg_wickets_last5']

In [95]:
model_df[['team1_avg_runs_last5']].head()

,team1_avg_runs_last5
0,158.727197
1,158.727197
2,158.727197
3,158.727197
4,158.727197


In [96]:
model_df[['team2_avg_runs_last5']].head()

,team2_avg_runs_last5
0,158.727197
1,158.727197
2,158.727197
3,82.000000
4,222.000000


In [97]:
'team1_avg_runs_last5' in model_df.columns

True

In [98]:
'team2_avg_runs_last5' in model_df.columns

True

In [99]:
'team1_avg_wickets_last5' in model_df.columns

True

In [100]:
'team2_avg_wickets_last5' in model_df.columns

True

In [101]:
model_df = model_df.copy()
model_df.reset_index(drop=True, inplace=True)

In [102]:
model_df['runs_diff'] = (
    model_df['team1_avg_runs_last5'].values
    - model_df['team2_avg_runs_last5'].values
)

model_df['wickets_diff'] = (
    model_df['team1_avg_wickets_last5'].values
    - model_df['team2_avg_wickets_last5'].values
)

In [103]:
model_df[
    [
        'team1',
        'team2',
        'team1_avg_runs_last5',
        'team2_avg_runs_last5',
        'runs_diff',
        'team1_avg_wickets_last5',
        'team2_avg_wickets_last5',
        'wickets_diff'
    ]
].head(10)

,team1,team2,team1_avg_runs_last5,team2_avg_runs_last5,runs_diff,team1_avg_wickets_last5,team2_avg_wickets_last5,wickets_diff
0,Kolkata Knight Riders,Royal Challengers Bengaluru,158.727197,158.727197,0.000000,5.848259,5.848259,0.000000
1,Chennai Super Kings,Punjab Kings,158.727197,158.727197,0.000000,5.848259,5.848259,0.000000
2,Rajasthan Royals,Delhi Capitals,158.727197,158.727197,0.000000,5.848259,5.848259,0.000000
3,Mumbai Indians,Royal Challengers Bengaluru,158.727197,82.000000,76.727197,5.848259,10.000000,-4.151741
4,Deccan Chargers,Kolkata Knight Riders,158.727197,222.000000,-63.272803,5.848259,3.000000,2.848259
5,Punjab Kings,Rajasthan Royals,207.000000,129.000000,78.000000,4.000000,8.000000,-4.000000
6,Deccan Chargers,Delhi Capitals,110.000000,132.000000,-22.000000,10.000000,1.000000,9.000000
7,Chennai Super Kings,Mumbai Indians,240.000000,165.000000,75.000000,5.000000,7.000000,-2.000000
8,Deccan Chargers,Rajasthan Royals,126.000000,148.500000,-22.500000,9.000000,6.000000,3.000000
9,Punjab Kings,Mumbai Indians,186.500000,183.500000,3.000000,6.000000,7.000000,-1.000000


In [104]:
model_df[
    [
        'elo_diff',
        'form_diff',
        'runs_diff',
        'wickets_diff',
        'team1_win'
    ]
].corr()

,elo_diff,form_diff,runs_diff,wickets_diff,team1_win
elo_diff,1.000000,0.615696,0.183950,-0.341955,0.050354
form_diff,0.615696,1.000000,0.243182,-0.561664,0.022572
runs_diff,0.183950,0.243182,1.000000,-0.238229,0.039366
wickets_diff,-0.341955,-0.561664,-0.238229,1.000000,-0.032460
team1_win,0.050354,0.022572,0.039366,-0.032460,1.000000


In [105]:
model_df.shape

(1169, 84)

In [107]:
model_df.columns

Index(['match_id', 'Unnamed: 0', 'date', 'match_type', 'event_name', 'innings',
       'batting_team_x', 'bowling_team', 'over', 'ball', 'ball_no', 'batter',
       'bat_pos', 'runs_batter', 'balls_faced', 'bowler', 'valid_ball',
       'runs_extras', 'runs_total', 'runs_bowler', 'runs_not_boundary',
       'extra_type', 'non_striker', 'non_striker_pos', 'wicket_kind',
       'player_out', 'fielders', 'runs_target', 'review_batter',
       'team_reviewed', 'review_decision', 'umpire', 'umpires_call',
       'player_of_match', 'match_won_by', 'win_outcome', 'toss_winner',
       'toss_decision', 'venue', 'city', 'day', 'month', 'year', 'season',
       'gender', 'team_type', 'superover_winner', 'result_type', 'method',
       'balls_per_over', 'overs', 'event_match_no', 'stage', 'match_number',
       'team_runs', 'team_balls', 'team_wicket', 'new_batter',
       'power_surge_start', 'batter_runs', 'batter_balls', 'bowler_wicket',
       'batting_partners', 'next_batter', 'striker_out',

In [108]:
model_df.to_csv(
    "../data/processed/final_model_df.csv",
    index=False
)

print("Saved Final Dataset")

Saved Final Dataset


In [109]:
model_df.columns.tolist()

['match_id',
 'Unnamed: 0',
 'date',
 'match_type',
 'event_name',
 'innings',
 'batting_team_x',
 'bowling_team',
 'over',
 'ball',
 'ball_no',
 'batter',
 'bat_pos',
 'runs_batter',
 'balls_faced',
 'bowler',
 'valid_ball',
 'runs_extras',
 'runs_total',
 'runs_bowler',
 'runs_not_boundary',
 'extra_type',
 'non_striker',
 'non_striker_pos',
 'wicket_kind',
 'player_out',
 'fielders',
 'runs_target',
 'review_batter',
 'team_reviewed',
 'review_decision',
 'umpire',
 'umpires_call',
 'player_of_match',
 'match_won_by',
 'win_outcome',
 'toss_winner',
 'toss_decision',
 'venue',
 'city',
 'day',
 'month',
 'year',
 'season',
 'gender',
 'team_type',
 'superover_winner',
 'result_type',
 'method',
 'balls_per_over',
 'overs',
 'event_match_no',
 'stage',
 'match_number',
 'team_runs',
 'team_balls',
 'team_wicket',
 'new_batter',
 'power_surge_start',
 'batter_runs',
 'batter_balls',
 'bowler_wicket',
 'batting_partners',
 'next_batter',
 'striker_out',
 'winner_historical_win_pct',
 '

In [110]:
model_df.to_csv(
    "../data/processed/final_model_df.csv",
    index=False
)

print("Final dataset saved")

Final dataset saved


In [111]:
h2h_history = {}

team1_h2h_win_pct = []
team2_h2h_win_pct = []

In [112]:
for _, row in model_df.iterrows():

    t1 = row['team1']
    t2 = row['team2']

    pair = tuple(sorted([t1, t2]))

    if pair not in h2h_history:
        h2h_history[pair] = {
            t1: 0,
            t2: 0,
            'matches': 0
        }

    history = h2h_history[pair]

    matches = history['matches']

    if matches == 0:
        team1_h2h_win_pct.append(0.5)
        team2_h2h_win_pct.append(0.5)
    else:
        team1_h2h_win_pct.append(
            history.get(t1, 0) / matches
        )

        team2_h2h_win_pct.append(
            history.get(t2, 0) / matches
        )

    winner = row['match_won_by']

    if winner in history:
        history[winner] += 1

    history['matches'] += 1

In [113]:
model_df['team1_h2h_win_pct'] = team1_h2h_win_pct
model_df['team2_h2h_win_pct'] = team2_h2h_win_pct

model_df['h2h_diff'] = (
    model_df['team1_h2h_win_pct']
    - model_df['team2_h2h_win_pct']
)

In [114]:
model_df[
    [
        'team1',
        'team2',
        'team1_h2h_win_pct',
        'team2_h2h_win_pct',
        'h2h_diff'
    ]
].tail(20)

,team1,team2,team1_h2h_win_pct,team2_h2h_win_pct,h2h_diff
1149,Gujarat Titans,Punjab Kings,0.500000,0.500000,0.000000
1150,Lucknow Super Giants,Delhi Capitals,0.428571,0.571429,-0.142857
1151,Sunrisers Hyderabad,Kolkata Knight Riders,0.344828,0.655172,-0.310345
1152,Chennai Super Kings,Punjab Kings,0.516129,0.483871,0.032258
1153,Mumbai Indians,Delhi Capitals,0.567568,0.432432,0.135135
1154,Rajasthan Royals,Gujarat Titans,0.250000,0.750000,-0.500000
1155,Sunrisers Hyderabad,Lucknow Super Giants,0.333333,0.666667,-0.333333
1156,Royal Challengers Bengaluru,Chennai Super Kings,0.382353,0.617647,-0.235294
1157,Rajasthan Royals,Mumbai Indians,0.466667,0.533333,-0.066667
1158,Gujarat Titans,Delhi Capitals,0.571429,0.428571,0.142857


In [115]:
model_df[
    [
        'team1_h2h_win_pct',
        'team2_h2h_win_pct'
    ]
].describe()

,team1_h2h_win_pct,team2_h2h_win_pct
count,1169.000000,1169.000000
mean,0.504011,0.495989
std,0.220553,0.220553
min,0.000000,0.000000
25%,0.400000,0.400000
50%,0.500000,0.500000
75%,0.600000,0.600000
max,1.000000,1.000000


In [116]:
venue_stats = (
    innings_summary
    .groupby('match_id')
    .agg(
        venue_runs=('total_runs', 'sum')
    )
    .reset_index()
)

venue_stats = venue_stats.merge(
    model_df[['match_id', 'venue']],
    on='match_id',
    how='left'
)

venue_stats.head()

,match_id,venue_runs,venue
0,335982,304,M Chinnaswamy Stadium
1,335983,447,"Punjab Cricket Association Stadium, Mohali"
2,335984,261,Feroz Shah Kotla
3,335985,331,Wankhede Stadium
4,335986,222,Eden Gardens


In [117]:
venue_avg_score = (
    venue_stats
    .groupby('venue')['venue_runs']
    .mean()
    .reset_index()
)

venue_avg_score.columns = [
    'venue',
    'venue_avg_score'
]

venue_avg_score.head()

,venue,venue_avg_score
0,Arun Jaitley Stadium,319.692308
1,"Arun Jaitley Stadium, Delhi",382.500000
2,Barabati Stadium,325.428571
3,"Barsapara Cricket Stadium, Guwahati",326.500000
4,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,340.608696


In [118]:
venue_match_count = (
    venue_stats
    .groupby('venue')
    .size()
    .reset_index(name='venue_match_count')
)

venue_match_count.head()

,venue,venue_match_count
0,Arun Jaitley Stadium,13
1,"Arun Jaitley Stadium, Delhi",24
2,Barabati Stadium,7
3,"Barsapara Cricket Stadium, Guwahati",8
4,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,23


In [119]:
model_df = model_df.merge(
    venue_avg_score,
    on='venue',
    how='left'
)

model_df = model_df.merge(
    venue_match_count,
    on='venue',
    how='left'
)

model_df[
    [
        'venue',
        'venue_avg_score',
        'venue_match_count'
    ]
].head()

,venue,venue_avg_score,venue_match_count
0,M Chinnaswamy Stadium,315.225806,62
1,"Punjab Cricket Association Stadium, Mohali",313.914286,35
2,Feroz Shah Kotla,311.271186,59
3,Wankhede Stadium,320.361111,72
4,Eden Gardens,307.246753,77


In [120]:
model_df.to_csv(
    "../data/processed/final_model_df_v2.csv",
    index=False
)

print("Final dataset saved")

Final dataset saved
